# 0.4 Bandwidth Limit Scenarios

Derive candidate byte-denominated bandwidth limits for `bandwidth = calldata + BAL bytes`, then map those byte caps into EIP-7999 resource gas limits. Targets are deliberately left for later fee-market tuning.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bandwidth_limits.eip7999_metering import BandwidthMeteringConfig
from bandwidth_limits.propagation import CONSERVATIVE_P90, EMPIRICAL_P90
from bandwidth_limits.scenarios import GLAMSTERDAM_NO_8279, GLAMSTERDAM_PLUS_8279
from bandwidth_limits.sweep import (
    add_propagation_times,
    best_strategy_sweep,
    eip7999_limit_candidates,
    historical_bandwidth_usage,
    safe_payload_cap_sweep,
)
from bandwidth_limits.worst_case import sweep_strategies

## Part 1: Worst-Case Strategy Sweep

In [ ]:
GAS_LIMITS = [60_000_000, 100_000_000, 150_000_000, 200_000_000, 300_000_000, 450_000_000]
SCHEDULES = [GLAMSTERDAM_NO_8279, GLAMSTERDAM_PLUS_8279]

all_strategies = sweep_strategies(GAS_LIMITS, SCHEDULES)
best_table = best_strategy_sweep(GAS_LIMITS, SCHEDULES)
best_display_cols = [
    "execution_gas_limit",
    "schedule",
    "best_strategy",
    "calldata_bytes",
    "bal_bytes",
    "tx_access_list_bytes",
    "total_payload_bytes",
    "total_payload_mib",
    "gas_used",
]
display(best_table[best_display_cols])
display(all_strategies[["execution_gas_limit", "schedule", "strategy", "total_payload_bytes", "gas_used", "is_best"]])

## Part 2: Propagation Safety

In [ ]:
best_with_propagation = add_propagation_times(
    best_table,
    fits=[EMPIRICAL_P90, CONSERVATIVE_P90],
    payload_col="total_payload_bytes",
)
display(best_with_propagation[best_display_cols + ["empirical_p90_ms", "conservative_p90_ms"]])

safe_caps = safe_payload_cap_sweep(
    windows_ms=[3000, 4000, 6000],
    safety_factors=[0.75, 1.0],
    fit=CONSERVATIVE_P90,
)
display(safe_caps)

## Part 3: Candidate EIP-7999 Bandwidth Gas Limits

In [ ]:
limit_candidates = eip7999_limit_candidates(
    safe_caps,
    gas_per_safe_byte=16,
)
display(limit_candidates)

## Part 4: Historical Replay Compatibility

In [ ]:
historical_path = PROJECT_ROOT / "data" / "xatu_calldata_50_blocks_22886891_22886940.csv"

if historical_path.exists():
    historical = pd.read_csv(historical_path)
    default_cap = int(
        safe_caps[
            (safe_caps["window_ms"] == 4000)
            & (safe_caps["safety_factor"] == 0.75)
        ]["safe_bandwidth_bytes"].iloc[0]
    )
    metering_config = BandwidthMeteringConfig(
        safe_bandwidth_bytes=default_cap,
        gas_per_safe_byte=16,
        bal_gas_per_byte=16,
    )
    historical_usage = historical_bandwidth_usage(historical, metering_config)
    display(historical_usage.head())
    display(historical_usage.describe())
else:
    print(f"No historical CSV found at {historical_path}")